# 14 — Development trials: logistic regression to MLP

**Purpose.** Document the relevant notebook 06 trials and re-evaluate their configurations under the report's common protocol. Results determine the retained candidate; an MLP win is not assumed.

Development questions:
1. Does metadata improve a word TF-IDF logistic-regression baseline?
2. Does a compact MLP help with the same A2 features?
3. Do character features plus department information help?
4. Do more permissive word features or a deeper network help?
5. Do prediction blends add value?
6. How does the retained 30k-word deep A2 MLP compare with a matched A2 LR and its character extension?

**Data and evaluation follow the report.** Use the recovered 14,439-posting development pool (693 frauds, 3,555 groups, 331 fraud groups), restoring the exact 5 repeats × 5 outer group-aware folds. Three grouped inner folds select hyperparameters by mean **PR-AUC, implemented as average precision (AP)**. Inner out-of-fold predictions select the fraud threshold by maximum F1. Each neural fit makes its own training-only grouped stopping split before a fixed-epoch refit. No outer assessment labels choose epochs, settings, blend weights or thresholds.

The main table reports mean and sample SD of outer-fold **PR-AUC, fraud precision, recall and F1**. PR-AUC is primary; precision/recall at a fixed 5% review budget are secondary. Accuracy is excluded. Related groups stay together. Class balancing is recomputed within each fit; SMOTE is not used.

**Historical provenance.** The pasted notebook 06 contains code/descriptions but no numerical outputs. It documents the candidate designs; it does not establish which one won in Colab. This notebook is a new, harmonised evaluation, not a reproduction of an undocumented Colab score. Original 06 used a single internal validation split, 5k words and validation-AP stopping. Here its candidates use the report's nested procedure with stopping/refit inside each training fit. The final 08/09 model keeps its original 30k-word, validation-loss checkpoint recipe. Random-row experiments and unverified Trial 5 numbers are excluded.

**Split recovery.** All 25 original outer signatures match. Original CSV file bytes could not be reproduced, so historical scores are not imported into this main table. The reserved 3,441-posting test set is not scored or used for selection here. Some original test IDs may have entered earlier work under a different split, so a project-wide claim of a completely untouched test is not supported.

**Environment.** Use Python 3.11.9 and install `requirements-14.txt` in a virtual environment. Open the complete repository. The setup cell automatically downloads the fixed training-input ZIP from the GitHub release, verifies its checksum and extracts it. For offline use, place the same ZIP in the repository root first. No other notebooks need to run first. The data ZIP is a release asset, not a Git-tracked dataset.

In [ ]:
from pathlib import Path
import gc, hashlib, importlib.metadata, json, os, tempfile, time
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import average_precision_score
from sklearn.utils.class_weight import compute_class_weight
from threadpoolctl import threadpool_limits
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/features.py').exists())
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.features import TARGET, GROUP, build_inputs, logistic_pipeline
from src.evaluation import make_folds, best_f1_threshold, score_summary, paired_comparison, fold_signature, validate_folds
from src.d1_a2_mlp import A2MLP
from src.d1_a2_character_mlp import A2CharacterMLP
from src import d1_sparse_nn as early
from src import d1_sparse_nn_tuning as followup

def restore_folds(frame, assignments, expected):
    """Restore saved positions by ID and enforce every original fold signature."""
    if not frame.job_id.is_unique or frame.job_id.isna().any():
        raise ValueError('Training job IDs must be unique and present.')
    if assignments.duplicated(['repeat', 'job_id']).any():
        raise ValueError('Duplicate saved assignments.')
    positions = pd.Series(np.arange(len(frame)), index=frame.job_id)
    folds = []
    for (r, k), rows in assignments.groupby(['repeat', 'fold'], sort=True):
        if not rows.job_id.isin(positions.index).all():
            raise ValueError('Unknown assignment IDs.')
        val = np.sort(positions.loc[rows.job_id].to_numpy())
        if not np.array_equal(rows.set_index('job_id').loc[frame.iloc[val].job_id, GROUP],
                              frame.iloc[val][GROUP]):
            raise ValueError('Saved groups differ from reconstructed groups.')
        fit = np.setdiff1d(np.arange(len(frame)), val)
        folds.append((int(r), int(k), fit, val))
    folds = validate_folds(folds, frame[TARGET], frame[GROUP])
    actual = {f'{r}:{k}': fold_signature(frame[TARGET], frame[GROUP], fit, val)
              for r, k, fit, val in folds}
    if actual != expected or len(folds) != 25:
        raise ValueError('The complete original 25 fold signatures must match.')
    return folds

REFERENCE = ROOT / 'data/processed/14_mlp_development/reference'
INPUT_URL = 'https://github.com/stellaS-xy/bt4012-fake-job-fraud/releases/download/notebook14-inputs-v1/14_mlp_training_inputs.zip'
INPUT_SHA256 = 'ad27db69f3106e121b345a3f39819560aa42175f6e9f9f9b2aa6bbc46aee4c84'
if not (REFERENCE / 'recovered_train.csv').exists():
    import urllib.request, zipfile, io
    local_inputs = (ROOT / '14_mlp_training_inputs.zip', ROOT / 'data/processed/14_mlp_training_inputs.zip')
    archive = next((p for p in local_inputs if p.is_file()), None)
    if archive is None:
        print('Downloading the fixed training inputs (13.2 MB)...', flush=True)
        with urllib.request.urlopen(INPUT_URL, timeout=120) as response:
            payload = response.read()
    else:
        payload = archive.read_bytes()
    if hashlib.sha256(payload).hexdigest() != INPUT_SHA256:
        raise ValueError('Training-input ZIP checksum mismatch; do not substitute another split.')
    allowed = {str((REFERENCE / name).relative_to(ROOT)) for name in
               ('recovered_train.csv', 'outer_assignments.csv', 'lightgbm_a2_2k_plan.json', 'recovery.json')}
    with zipfile.ZipFile(io.BytesIO(payload)) as bundle:
        if set(bundle.namelist()) != allowed or len(bundle.namelist()) != len(allowed):
            raise ValueError('Unexpected training-input archive contents.')
        for name in allowed:
            target = ROOT / name
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(bundle.read(name))
    del payload

OUTPUT = ROOT / 'data/processed/14_mlp_development'
plan = json.loads((REFERENCE / 'lightgbm_a2_2k_plan.json').read_text())
recovery = json.loads((REFERENCE / 'recovery.json').read_text())
VERSIONS = {d.metadata['Name'].lower().replace('_', '-'): d.version
            for d in importlib.metadata.distributions()}
for package, expected in plan['versions'].items():
    if VERSIONS.get(package) != expected:
        raise RuntimeError(f'Select the requirements-14 interpreter: {package} needs {expected}, found {VERSIONS.get(package)}.')
for name in ('features.py', 'evaluation.py'):
    expected = plan['source_sha256'][name]
    assert hashlib.sha256((ROOT / 'src' / name).read_bytes()).hexdigest() == expected, name
train_bytes = (REFERENCE / 'recovered_train.csv').read_bytes()
assert hashlib.sha256(train_bytes).hexdigest() == recovery['recovered_training_file_sha256']
frame = pd.read_csv(REFERENCE / 'recovered_train.csv')
assignment_bytes = (REFERENCE / 'outer_assignments.csv').read_bytes()
assert hashlib.sha256(assignment_bytes).hexdigest() == plan['assignments_sha256']
assignments = pd.read_csv(REFERENCE / 'outer_assignments.csv')
folds = restore_folds(frame, assignments, plan['outer_signatures'])
assert (len(frame), int(frame[TARGET].sum()), frame[GROUP].nunique(),
        frame.loc[frame[TARGET].eq(1), GROUP].nunique()) == (14439, 693, 3555, 331)
X = build_inputs(frame.copy())
X['department'] = frame['department']
assert TARGET not in X
y, groups = frame[TARGET].to_numpy(), frame[GROUP].to_numpy()
print(f'Validated {len(frame):,} development postings, {int(y.sum())} frauds, all {len(folds)} original folds.')
print('No reserved test predictions; existing train/test files remain unchanged.')

## Trial plan and interpretation

Configurations are fixed before this run. A positive result is not required. The rich features change both character information and department encoding, so their contributions cannot be separated without another ablation. The deep rich trial changes architecture/dropout together. Moving from that trial to the retained 30k A2 recipe changes vocabulary, features and training settings together: compare complete pipelines and avoid attributing the difference to depth alone.

The two blends reuse genuinely inner-out-of-fold component predictions. Candidate/weight selection maximises mean **inner PR-AUC**, rather than 06's original validation-F1 blend selection, to follow the report. F1 is used only for the operating threshold. Blend weights never use outer scores. The exact candidate dependencies are listed in code.

In [ ]:
TRIALS = [
    ('text_lr_5k', 'Word TF-IDF LR (5k)', 'Text-only baseline'),
    ('a2_lr_5k', 'Text + metadata LR (5k)', 'Add A2 metadata to the text baseline'),
    ('compact_a2_5k', 'Compact A2 MLP (128/32, 5k)', 'Nonlinear classifier; same inputs as A2 LR 5k'),
    ('compact_rich_5k', 'Compact rich MLP (5k + char + department)', 'Add 15k character terms and department'),
    ('lr_mlp_blend', 'Inner-selected LR + compact/rich MLP blend', 'Complementary LR and neural scores'),
    ('relaxed_compact_rich_5k', 'Relaxed-word compact rich MLP', 'min_df=2, max_df=.95, no word stop list, ordinary TF'),
    ('deep_rich_5k', 'Relaxed-word deep rich MLP (512/256/128/64)', 'Deeper network on the same relaxed rich inputs'),
    ('mlp_followup_blend', 'Inner-selected original/new MLP blend', 'Combine original rich MLP with relaxed compact/deep MLP'),
    ('a2_lr_30k', 'A2 logistic regression (30k)', 'Matched baseline for the retained 08/09 pipeline'),
    ('deep_a2_30k', 'A2 deep MLP (30k; retained 08/09 recipe)', 'Original grouped MLP checkpoint/refit procedure'),
    ('deep_char_a2_30k', 'A2 + character deep MLP (30k + 10k)', 'Controlled character extension of retained MLP'),
]
LABELS = {key: name for key, name, _ in TRIALS}
display(pd.DataFrame(TRIALS, columns=['key', 'experiment', 'question']))

# Run All evaluates all trials on the complete 25-fold protocol.
# Set False only to inspect existing outputs without fitting.
RUN_TRAINING = True
OUTER_LIMIT = None  # None = all 25. Any smaller run is explicitly preliminary.
SELECTED_TRIALS = tuple(LABELS)
THREADS = 2

## Neural trials adapted to nested grouped evaluation

For the 06-style compact/deep-rich candidates, checkpoint epochs maximise AP on a separate grouped stopping split inside each call to `fit`. The stopping model/vocabulary are discarded. Fresh preprocessing and a fresh network then fit all rows supplied to `fit` for the selected number of epochs. Inner validation and outer scoring groups never select those epochs. Class weights use only the respective fitting labels.

The final deep A2 and character A2 candidates reuse the existing 08/09 estimators unchanged; they select checkpoint duration by stopping loss. Every candidate still uses the same three inner folds for threshold selection and the same saved outer folds for assessment. Different checkpoint objectives are explicit pipeline differences.

In [ ]:
class DevelopmentMLP(ClassifierMixin, BaseEstimator):
    def __init__(self, rich=False, relaxed=False, deep=False, word_features=5000,
                 max_epochs=20, patience=3, batch_size=128, seed=42):
        self.rich, self.relaxed, self.deep = rich, relaxed, deep
        self.word_features = word_features
        self.max_epochs, self.patience = max_epochs, patience
        self.batch_size, self.seed = batch_size, seed

    def _config(self):
        return early.NeuralConfig(word_features=self.word_features, character_features=15000,
            max_epochs=self.max_epochs, patience=self.patience, batch_size=self.batch_size)

    def _features(self):
        if self.relaxed:
            if not self.rich:
                raise ValueError('The historical relaxed trial uses rich features.')
            return followup.relaxed_features(self._config())
        return early.feature_builders(self._config())['rich' if self.rich else 'a2']

    def fit(self, inputs, labels):
        labels = np.asarray(labels, dtype=int)
        if self.seed != 42 or len(inputs) != len(labels) or set(np.unique(labels)) != {0, 1}:
            raise ValueError('Need seed 42, aligned labels and both classes.')
        _, _, fitting, stopping = make_folds(inputs[GROUP], labels, 5, 1, self.seed)[0]
        self.checkpoint_indices_ = (fitting.copy(), stopping.copy())
        cfg = self._config()
        factory = followup.DeepKerasAdapter if self.deep else early.KerasSparseAdapter
        features = self._features()
        a = sparse.csr_matrix(features.fit_transform(inputs.iloc[fitting], labels[fitting]), dtype=np.float32)
        b = sparse.csr_matrix(features.transform(inputs.iloc[stopping]), dtype=np.float32)
        _, _, detail = early.fit_neural((a, b, b), labels[fitting], labels[stopping], cfg,
                                       seed=self.seed, adapter_factory=factory)
        self.selected_epoch_ = int(detail['selected_epoch'])
        self.checkpoint_history_ = detail['history']
        del features, a, b
        gc.collect()
        self.features_ = self._features()
        matrix = sparse.csr_matrix(self.features_.fit_transform(inputs, labels), dtype=np.float32)
        self.adapter_ = factory(matrix.shape[1], cfg, self.seed)
        weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=labels)[labels]
        generator = np.random.default_rng(self.seed)
        print(f'  Refit {"deep" if self.deep else "compact"} MLP: {len(labels)} rows, {self.selected_epoch_} epochs.', flush=True)
        for _ in range(self.selected_epoch_):
            self.adapter_.reset_metrics()
            order = generator.permutation(len(labels))
            for start in range(0, len(labels), self.batch_size):
                idx = order[start:start + self.batch_size]
                self.adapter_.train_batch(matrix[idx], labels[idx], weights[idx])
        self.classes_ = np.array([0, 1])
        return self

    def predict_proba(self, inputs):
        matrix = sparse.csr_matrix(self.features_.transform(inputs), dtype=np.float32)
        p = early._predict_batches(self.adapter_, matrix, self.batch_size)
        return np.column_stack([1 - p, p])


def candidate_spec(key):
    if key in ('text_lr_5k', 'a2_lr_5k', 'a2_lr_30k'):
        blocks = ['tfidf'] if key == 'text_lr_5k' else ['raw', 'engineered', 'text_style', 'tfidf']
        limit = 30000 if key == 'a2_lr_30k' else 5000
        return logistic_pipeline(blocks, tfidf_max_features=limit), {'model__C': [.1, .3, 1.]}
    if key == 'deep_a2_30k':
        return A2MLP(), {}
    if key == 'deep_char_a2_30k':
        return A2CharacterMLP(), {}
    choices = {'compact_a2_5k': {}, 'compact_rich_5k': {'rich': True},
               'relaxed_compact_rich_5k': {'rich': True, 'relaxed': True},
               'deep_rich_5k': {'rich': True, 'relaxed': True, 'deep': True}}
    if key not in choices:
        raise ValueError(f'Unknown base trial: {key}')
    return DevelopmentMLP(**choices[key]), {}


BLENDS = {
    'lr_mlp_blend': ('a2_lr_5k', ('compact_a2_5k', 'compact_rich_5k')),
    'mlp_followup_blend': ('compact_rich_5k', ('relaxed_compact_rich_5k', 'deep_rich_5k')),
}
BASE_TRIALS = tuple(k for k in LABELS if k not in BLENDS)

## Resumable fitting and training-only selection

Every completed inner fit and outer prediction is saved under an experiment fingerprint containing data, notebook code, source and package hashes. Expanding the requested fold count reuses matching stages. Changing the model code/settings creates a new run. Blend trials reuse component predictions without extra fitting.

For each LR setting, select the highest mean inner-fold AP. For a blend, select its candidate/weight by the same mean inner-fold AP rule. Choose the operating threshold from the selected pipeline's inner OOF predictions. Report only held-out outer scores as performance. The repeated outer folds remain development evidence after earlier exploration.

In [ ]:
MAIN_METRICS = ('pr_auc', 'precision', 'recall', 'f1')
REVIEW_METRICS = ('precision_top5pct', 'recall_top5pct')

def atomic_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    tmp.write_text(json.dumps(value, indent=2, allow_nan=False))
    tmp.replace(path)

def notebook_code_hash():
    nb = json.loads((ROOT / 'notebooks/14_d1_mlp_development.ipynb').read_text())
    import ast
    source = '\n'.join(''.join(c['source']) for c in nb['cells'] if c['cell_type'] == 'code')
    tree = ast.parse(source)
    controls = {'RUN_TRAINING', 'OUTER_LIMIT', 'SELECTED_TRIALS', 'THREADS'}
    tree.body = [node for node in tree.body if not (isinstance(node, ast.Assign)
        and any(isinstance(t, ast.Name) and t.id in controls for t in node.targets))]
    return hashlib.sha256(ast.dump(tree, include_attributes=False).encode()).hexdigest()

def experiment_context():
    source_names = ('features.py', 'evaluation.py', 'd1_a2_mlp.py', 'd1_row_neural.py',
                    'd1_sparse_nn.py', 'd1_sparse_nn_tuning.py', 'd1_a2_character_mlp.py')
    return {'training_sha256': recovery['recovered_training_file_sha256'],
        'outer_signatures': plan['outer_signatures'], 'versions': VERSIONS,
        'sources': {name: hashlib.sha256((ROOT / 'src' / name).read_bytes()).hexdigest() for name in source_names},
        'notebook_code_sha256': notebook_code_hash(),
        'base_specs': {k: {'estimator': repr(candidate_spec(k)[0]), 'grid': candidate_spec(k)[1]} for k in BASE_TRIALS},
        'blends': BLENDS, 'weights': [0., .25, .5, .75, 1.], 'inner_splits': 3,
        'threads': THREADS, 'seed_rule': '42 + repeat*1000 + fold', 'threshold': 'inner OOF maximum F1',
        'primary_metric': 'PR-AUC implemented as average precision', 'expected_folds': 25,
        'reserved_test_predictions_computed': False}

def load_npz(path):
    with np.load(path, allow_pickle=False) as saved:
        return {key: saved[key].copy() for key in saved.files}

def save_npz(path, **values):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix('.tmp')
    with tmp.open('wb') as stream:
        np.savez_compressed(stream, **values)
    tmp.replace(path)

def stage(model, fitting, scoring, path):
    if set(groups[fitting]) & set(groups[scoring]):
        raise ValueError('Training/scoring group overlap.')
    if path.exists():
        saved = load_npz(path)
        if not (np.array_equal(saved['fitting_ids'], frame.iloc[fitting].job_id) and
                np.array_equal(saved['scoring_ids'], frame.iloc[scoring].job_id)):
            raise ValueError('Cached stage row identity mismatch.')
        return saved['p']
    fitted = clone(model).fit(X.iloc[fitting], y[fitting])
    p = fitted.predict_proba(X.iloc[scoring])[:, 1]
    score_summary(y[scoring], p)  # Validate probabilities; do not choose a cutoff here.
    detail = {'selected_epoch': getattr(fitted, 'selected_epoch_', None),
              'checkpoint_history': getattr(fitted, 'checkpoint_history_', [])}
    save_npz(path, p=p, fitting_ids=frame.iloc[fitting].job_id.to_numpy(),
        scoring_ids=frame.iloc[scoring].job_id.to_numpy(), detail=json.dumps(detail))
    framework = getattr(getattr(fitted, 'adapter_', None), 'tf', None)
    del fitted
    if framework is not None:
        framework.keras.backend.clear_session(free_memory=True)
    gc.collect()
    return p

def base_prediction(key, fold, directory):
    r, k, fitting, scoring = fold
    path = directory / key / f'r{r}_f{k}.npz'
    if path.exists():
        saved = load_npz(path)
        if not (np.array_equal(saved['training_ids'], frame.iloc[fitting].job_id) and
                np.array_equal(saved['job_id'], frame.iloc[scoring].job_id)):
            raise ValueError('Completed fold cache IDs differ.')
        return saved
    started = time.monotonic()
    model, grid = candidate_spec(key)
    inner = make_folds(groups[fitting], y[fitting], 3, 1, 42 + r*1000 + k)
    best_ap, best_params, best_oof = -np.inf, None, None
    diagnostics = []
    for params in ParameterGrid(grid):
        tag = hashlib.sha256(json.dumps(params, sort_keys=True).encode()).hexdigest()[:12]
        oof = np.full(len(fitting), np.nan)
        aps = []
        for _, number, a, b in inner:
            print(f'  {key}: inner {number+1}/3, settings {params}', flush=True)
            p = stage(clone(model).set_params(**params), fitting[a], fitting[b],
                      directory / key / f'r{r}_f{k}_stages' / f'{tag}_inner{number}.npz')
            oof[b] = p
            aps.append(float(average_precision_score(y[fitting[b]], p)))
        mean = float(np.mean(aps))
        diagnostics.append({'params': params, 'mean_inner_pr_auc': mean})
        if mean > best_ap:
            best_ap, best_params, best_oof = mean, params, oof.copy()
    if not np.isfinite(best_oof).all():
        raise ValueError('Incomplete inner OOF predictions.')
    threshold = best_f1_threshold(y[fitting], best_oof)
    p = stage(clone(model).set_params(**best_params), fitting, scoring,
              directory / key / f'r{r}_f{k}_stages' / 'outer.npz')
    detail = {'params': best_params, 'threshold': threshold, 'inner_candidates': diagnostics,
              'minutes': (time.monotonic() - started)/60}
    save_npz(path, p=p, oof=best_oof, job_id=frame.iloc[scoring].job_id.to_numpy(),
             training_ids=frame.iloc[fitting].job_id.to_numpy(), detail=json.dumps(detail))
    return load_npz(path)

def blend_prediction(key, fold, components, directory):
    r, k, fitting, scoring = fold
    reference, candidates = BLENDS[key]
    inner = make_folds(groups[fitting], y[fitting], 3, 1, 42 + r*1000 + k)
    diagnostics, best_ap, best = [], -np.inf, None
    # Stable ordering: original/zero-weight configuration wins exact ties.
    for weight in (0., .25, .5, .75, 1.):
        for candidate in candidates:
            oof = (1-weight)*components[reference]['oof'] + weight*components[candidate]['oof']
            mean = float(np.mean([average_precision_score(y[fitting[b]], oof[b]) for _, _, _, b in inner]))
            diagnostics.append({'candidate': candidate, 'weight': weight, 'mean_inner_pr_auc': mean})
            if mean > best_ap:
                best_ap, best = mean, (candidate, weight, oof)
    candidate, weight, oof = best
    threshold = best_f1_threshold(y[fitting], oof)
    p = (1-weight)*components[reference]['p'] + weight*components[candidate]['p']
    detail = {'params': {'reference': reference, 'candidate': candidate, 'weight': weight},
              'threshold': threshold, 'inner_candidates': diagnostics, 'minutes': 0.}
    path = directory / key / f'r{r}_f{k}.npz'
    save_npz(path, p=p, oof=oof, job_id=frame.iloc[scoring].job_id.to_numpy(),
             training_ids=frame.iloc[fitting].job_id.to_numpy(), detail=json.dumps(detail))
    return load_npz(path)

## Run or inspect matching saved results

**Run All starts training** (`RUN_TRAINING=True`). Set it to `False` only to inspect existing results. The default requests **all 25 folds**; a smaller `OUTER_LIMIT` is an explicitly partial screen. This is substantially more work than notebook 06's one-fold screen. Avoid running other expensive experiments on the same machine during this run. Individual completed inner stages resume after interruption.

No scores from the 14,616-posting split or notebook 06's single-validation procedure enter the new performance table. Completed folds are compared using the intersection shared by **all selected trials**, avoiding a comparison of unequal fold subsets.

In [ ]:
def required_bases(keys):
    needed = set(keys) - set(BLENDS)
    for key in set(keys) & set(BLENDS):
        reference, candidates = BLENDS[key]
        needed.update((reference, *candidates))
    return [key for key in BASE_TRIALS if key in needed]

def checkpoint_reports(rows, directory, keys):
    scores = pd.DataFrame(rows)
    scores.to_csv(directory / 'fold_metrics.csv', index=False)
    counts = {key: sum(row['key'] == key for row in rows) for key in keys}
    atomic_json(directory / 'progress.json', {'completed_folds': counts, 'expected_folds': 25,
        'complete': all(value == 25 for value in counts.values()), 'updated_utc': pd.Timestamp.now(tz='UTC').isoformat()})

def run_development(directory, keys, outer_limit=None):
    if len(set(keys)) != len(keys) or not keys or set(keys) - set(LABELS):
        raise ValueError('Select unique supported trials.')
    if outer_limit is not None and (not isinstance(outer_limit, int) or not 1 <= outer_limit <= 25):
        raise ValueError('OUTER_LIMIT must be None or 1..25.')
    requested = folds if outer_limit is None else folds[:outer_limit]
    rows = []
    for fold in requested:
        r, k, fitting, scoring = fold
        print(f'Repeat {r+1}/5, outer fold {k+1}/5', flush=True)
        components = {}
        with threadpool_limits(limits=THREADS):
            for key in required_bases(keys):
                components[key] = base_prediction(key, fold, directory)
                if key in keys:
                    saved = components[key]
                    detail = json.loads(saved['detail'].item())
                    measured = score_summary(y[scoring], saved['p'], threshold=detail['threshold'])
                    rows.append({'key': key, 'model': LABELS[key], 'repeat': r, 'fold': k,
                        'fold_signature': plan['outer_signatures'][f'{r}:{k}'],
                        'train_rows': len(fitting), 'val_rows': len(scoring), **measured,
                        'selected_params': json.dumps(detail['params']), 'minutes': detail['minutes']})
                    checkpoint_reports(rows, directory, keys)
            for key in keys:
                if key not in BLENDS:
                    continue
                saved = blend_prediction(key, fold, components, directory)
                detail = json.loads(saved['detail'].item())
                rows.append({'key': key, 'model': LABELS[key], 'repeat': r, 'fold': k,
                    'fold_signature': plan['outer_signatures'][f'{r}:{k}'],
                    'train_rows': len(fitting), 'val_rows': len(scoring),
                    **score_summary(y[scoring], saved['p'], threshold=detail['threshold']),
                    'selected_params': json.dumps(detail['params']), 'minutes': detail['minutes']})
                checkpoint_reports(rows, directory, keys)
    return pd.DataFrame(rows)

from contextlib import contextmanager

@contextmanager
def training_lock(path):
    """Prevent duplicate workers on macOS/Linux and Windows."""
    with path.open('a+b') as handle:
        if os.name == 'nt':
            import msvcrt
            handle.seek(0, 2)
            if handle.tell() == 0:
                handle.write(b'0')
                handle.flush()
            handle.seek(0)
            try:
                msvcrt.locking(handle.fileno(), msvcrt.LK_NBLCK, 1)
            except OSError as error:
                raise RuntimeError('Notebook 14 is already running in this results folder.') from error
            try:
                yield
            finally:
                handle.seek(0)
                msvcrt.locking(handle.fileno(), msvcrt.LK_UNLCK, 1)
        else:
            import fcntl
            try:
                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError as error:
                raise RuntimeError('Notebook 14 is already running in this results folder.') from error
            try:
                yield
            finally:
                fcntl.flock(handle, fcntl.LOCK_UN)

context = experiment_context()
fingerprint = hashlib.sha256(json.dumps(context, sort_keys=True).encode()).hexdigest()[:20]
directory = OUTPUT / fingerprint
if RUN_TRAINING:
    directory.mkdir(parents=True, exist_ok=True)
    with training_lock(directory / 'run.lock'):
        import tensorflow as tf
        try:
            tf.config.threading.set_intra_op_parallelism_threads(THREADS)
            tf.config.threading.set_inter_op_parallelism_threads(1)
        except RuntimeError:
            pass  # A reused notebook kernel may already have initialised TensorFlow.
        atomic_json(directory / 'manifest.json', context)
        scores = run_development(directory, SELECTED_TRIALS, OUTER_LIMIT)
elif (directory / 'fold_metrics.csv').exists():
    scores = pd.read_csv(directory / 'fold_metrics.csv')
else:
    scores = pd.DataFrame()
    print('No new matched results yet. RUN_TRAINING=False; historical scores are not substituted.')
print('Experiment directory:', directory)

In [ ]:
def matching_tables(scores, keys):
    if scores.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    selected = scores.loc[scores.key.isin(keys)]
    identities = [set(zip(selected.loc[selected.key.eq(key), 'repeat'],
                          selected.loc[selected.key.eq(key), 'fold'])) for key in keys]
    common = sorted(set.intersection(*identities))
    if not common:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    matched = selected.merge(pd.DataFrame(common, columns=['repeat', 'fold']))
    if matched.duplicated(['key', 'repeat', 'fold']).any():
        raise ValueError('Duplicate model/fold result.')
    summary = []
    for key in keys:
        table = matched.loc[matched.key.eq(key)]
        summary.append({'key': key, 'model': LABELS[key], 'matching_folds': len(common),
            'complete': len(common) == 25,
            **{m: table[m].mean() for m in MAIN_METRICS + REVIEW_METRICS},
            **{m + '_sd': table[m].std(ddof=1) for m in MAIN_METRICS}})
    paired = []
    if len(common) >= 2 and 'a2_lr_30k' in keys:
        baseline = matched.loc[matched.key.eq('a2_lr_30k')]
        for key in keys:
            if key == 'a2_lr_30k':
                continue
            for metric in MAIN_METRICS:
                paired.append({'candidate': LABELS[key], 'reference': LABELS['a2_lr_30k'],
                    'metric': metric, **paired_comparison(matched.loc[matched.key.eq(key)], baseline, metric)})
    return pd.DataFrame(summary).sort_values('pr_auc', ascending=False), pd.DataFrame(paired), matched

summary, paired, matched = matching_tables(scores, SELECTED_TRIALS)
if summary.empty:
    print('Performance table pending: no folds completed by every selected candidate.')
else:
    display(summary[['model', 'matching_folds', 'complete', *MAIN_METRICS,
                     *[m + '_sd' for m in MAIN_METRICS]]].round(4))
    print('Secondary review-budget measures:')
    display(summary[['model', *REVIEW_METRICS]].round(4))
    if not paired.empty:
        display(paired.round(4))
    summary.to_csv(directory / 'matched_summary.csv', index=False)
    paired.to_csv(directory / 'paired_differences.csv', index=False)
    (directory / 'matched_report.html').write_text(
        '<!doctype html><meta charset="utf-8"><h1>14: matched development trials</h1>'
        '<p>PR-AUC = AP. Same recovered report split and saved folds. Partial results are provisional.</p>'
        + summary.to_html(index=False) + paired.to_html(index=False))

## Historical evidence for why MLP was retained before this rerun

The following is a clearly labelled record from the saved **08/09 five-fold experiment on the different 14,616-posting / 659-fraud development split**. It supports the earlier decision to investigate MLP. It is not part of the new report-split ranking above and does not fill any missing Colab 06 outputs.

| Historical configuration | Mean PR-AUC (AP) | Interpretation within that experiment |
|---|---:|---|
| A2 logistic regression | 0.5914 | Matched linear baseline |
| A2 deep MLP | 0.6776 | AP improved on all five folds versus A2 LR |
| A2 + character deep MLP | 0.6638 | Character extension did not improve mean AP |

Source: saved outputs in `08_d1_a2_grouped_mlp.ipynb` and the five-fold checkpoint table in `reports/tables/09_best_model/model_comparison.csv`. Five folds are not a completed 25-fold repeated run. Earlier code and data versions remain distinct from the new results.

## Final interpretation — generated from the completed comparison

Use the table to describe which changes helped, did not help, or remained inconclusive. Paired intervals use an approximate repeated-CV correction; overlapping folds and earlier experimentation limit the strength of a superiority claim. No result here measures production performance.

The highest mean AP identifies the primary-metric leader. Review variability and fraud precision/recall/F1 before recording the final pipeline choice. Do not revise weights or thresholds using outer answers to make MLP win. If the table favours another candidate, report it. A development selection is separate from final held-out assessment.

In [ ]:
if summary.empty or not summary.complete.all():
    print('FINAL MODEL: pending — every selected candidate must complete all 25 matching folds.')
else:
    leader = summary.iloc[0]
    print(f'Highest mean development PR-AUC: {leader["model"]} ({leader.pr_auc:.4f}).')
    print('This is the ranking leader, not an automatic claim of superiority or production readiness.')
    comparisons = [
        ('a2_lr_5k', 'text_lr_5k', 'Adding metadata to LR'),
        ('compact_a2_5k', 'a2_lr_5k', 'Replacing LR with a compact MLP'),
        ('compact_rich_5k', 'compact_a2_5k', 'Adding character and department features'),
        ('relaxed_compact_rich_5k', 'compact_rich_5k', 'Relaxing word-vectorizer settings'),
        ('deep_rich_5k', 'relaxed_compact_rich_5k', 'Changing to the deeper rich network'),
        ('deep_a2_30k', 'a2_lr_30k', 'Retained deep A2 MLP versus matched LR'),
        ('deep_char_a2_30k', 'deep_a2_30k', 'Adding characters to retained deep A2 MLP'),
    ]
    indexed = summary.set_index('key')
    for candidate, reference, description in comparisons:
        if candidate in indexed.index and reference in indexed.index:
            difference = indexed.loc[candidate, 'pr_auc'] - indexed.loc[reference, 'pr_auc']
            print(f'{description}: mean PR-AUC change {difference:+.4f}.')
    print('Final selected pipeline and justification: [record after reviewing the complete matched table].')